# 19 — MMR redundancy ablation

이 노트북은 **Codex coder agent**가 `skn25` fresh kernel에서 실행한 개발셋 단일 실험이다. 저장된 16번 `vector_0.4_bm25_0.6` 후보와 기존 chunk embedding만 사용하며 GPU, 모델/custom code, network/API, 새 embedding, Chroma query를 사용하지 않는다. Holdout 및 운영 일반화를 평가하지 않는다.

고정 비교는 depth Top20/Top50, MMR λ 0.7/0.8/0.9, no-MMR baseline이며 primary는 Top50, final ranking 평가는 Top5다. Top20 relevance normalization도 query별 Top50 `total_score` min/max를 anchor로 쓴다. 첫 선택은 fused rank 1이고 이후 `λ*rel_norm-(1-λ)*max(0, cosine)`으로 선택한다. 1e-12 tie 안에서는 rel_norm 내림차순, 원래 rank 오름차순, chunk ID 오름차순이다.


In [1]:
from pathlib import Path
import csv, hashlib, itertools, json, math, os, statistics, tempfile, unicodedata
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (PROJECT_ROOT / 'notebooks').is_dir()
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/19_mmr_redundancy_ablation'; OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_13 = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
SOURCE_16 = PROJECT_ROOT / 'notebooks/data/16_normalized_rrf_weight_ablation'
SOURCE_18 = PROJECT_ROOT / 'notebooks/data/18_answer_bearing_grouped_retrieval_reevaluation'
CACHE_ROOT = SOURCE_13 / 'embedding_cache/text-embedding-3-small'
CACHE_NAMES = ('3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4.npz','335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba.npz','93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f.npz','b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2.npz','8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0.npz','cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55.npz')
INPUTS = {
 'chunks_13': SOURCE_13/'chunks.jsonl', 'queries_13': SOURCE_13/'retrieval_per_query.csv', 'embedding_usage_13': SOURCE_13/'embedding_usage.json',
 'candidates_16': SOURCE_16/'rrf_weight_candidates.csv', 'candidate_recall_16': SOURCE_16/'rrf_weight_candidate_recall.csv', 'summary_16': SOURCE_16/'rrf_weight_ablation_summary.json', 'manifest_16': SOURCE_16/'rrf_weight_run_manifest.json',
 'contract_18': SOURCE_18/'evaluation_contract.json', 'relevance_18': SOURCE_18/'relevance_sets.jsonl', 'exact_groups_18': SOURCE_18/'exact_document_groups.jsonl', 'per_query_18': SOURCE_18/'per_query_metrics.csv', 'summary_csv_18': SOURCE_18/'summary.csv', 'summary_json_18': SOURCE_18/'summary.json', 'coverage_18': SOURCE_18/'candidate_coverage.csv',
 **{f'embedding_cache_{i+1}': CACHE_ROOT/name for i,name in enumerate(CACHE_NAMES)},
}
DEPTHS=(20,50); LAMBDAS=(0.7,0.8,0.9); PRIMARY_DEPTH=50; FINAL_K=5; TOLERANCE=1e-12; SPAN_EPS=1e-15
VIEWS=('strict_raw','answer_bearing_raw','strict_exact_doc_dedup','answer_bearing_exact_doc_dedup','answer_bearing_gold_family_oracle')
GROUPS=('all','card','evidence','numeric','semantic'); DENOMINATORS={'all':30,'card':10,'evidence':20,'numeric':10,'semantic':10}
METRICS=('card_hit_at_3','hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5')
SYSTEMS=('no_mmr',)+tuple(f'mmr_{value:.1f}' for value in LAMBDAS)

CONTRACT = {
 'schema_version':'mmr_redundancy_ablation_v1', 'provenance':'Codex coder agent', 'scope':{'development_only':True,'single_run':True,'holdout_used':False},
 'input':{'configuration':'vector_0.4_bm25_0.6','depths':list(DEPTHS),'cached_chunk_embeddings_only':True},
 'mmr':{'lambdas':list(LAMBDAS),'primary_depth':PRIMARY_DEPTH,'final_k':FINAL_K,'top20_uses_top50_minmax_anchor':True,'span_failure_threshold':SPAN_EPS,'embedding_math':'float64 unit-normalized cosine clipped to [-1,1]','redundancy_penalty':'max(0, cosine)','first_selection':'fused_rank_1','formula':'lambda*rel_norm-(1-lambda)*max_positive_similarity','tie_tolerance':TOLERANCE,'tie_break':['rel_norm_desc','original_rank_asc','chunk_id_asc'],'full_depth_permutation':True,'lambda_1_must_restore_fused_rank':True},
 'evaluation':{'views':list(VIEWS),'groups':DENOMINATORS,'raw_and_exact_dedup_top5':True,'leaf_only_excluded':True,'selective_oracle_excluded':True},
 'containment_definition':'for same-card cross-level pairs, either non-empty NFKC/lower/whitespace-normalized document is an exact substring of the other',
 'selection_gate':{
  'hard_relevance':[
   'strict_raw evidence20 Card Hit@3/Hit@3/Recall@5/MRR@5/nDCG@5 all non-regressing and MRR or nDCG strictly improving',
   'answer_bearing_raw evidence20 Hit@3/MRR@5 non-regressing',
   'strict_exact_doc_dedup evidence20 Hit@3/MRR@5/nDCG@5 non-regressing',
   'answer_bearing_exact_doc_dedup evidence20 Hit@3/MRR@5/nDCG@5 non-regressing',
   'answer_bearing_gold_family_oracle evidence20 Hit@3/MRR@5/nDCG@5 non-regressing',
   'card10 Card Hit@3 non-regressing', 'evidence20 expected-card share@5 non-regressing'],
  'diversity':['evidence20 exact duplicate excess non-increasing','evidence20 cross-level containment non-increasing','positive pairwise cosine mean or containment strictly decreasing','redundancy W/L/T improved > worsened'],
  'tie_break':['strict_raw evidence nDCG desc','strict_raw evidence MRR desc','evidence expected-card share desc','evidence containment asc','evidence positive cosine mean asc','higher lambda'], 'fallback':'retain_no_mmr'},
 'execution':{'environment':'skn25','fresh_kernel':True,'cpu_only':True,'network_api_calls':0,'gpu_calls':0,'model_custom_code_calls':0,'new_embedding_calls':0,'chroma_queries':0,'package_installs':0},
}
assert os.environ.get('CUDA_VISIBLE_DEVICES','') == '' and os.environ.get('HF_HUB_OFFLINE') == '1' and os.environ.get('TRANSFORMERS_OFFLINE') == '1'
def sha256_file(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def sha256_text(value): return hashlib.sha256(value.encode()).hexdigest()
def normalize(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
def read_csv(path): return list(csv.DictReader(path.open(encoding='utf-8',newline='')))
def read_jsonl(path): return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
def write_json(path,value):
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',dir=path.parent,delete=False) as handle: temporary=Path(handle.name); json.dump(value,handle,ensure_ascii=False,indent=2); handle.write('\n')
    os.replace(temporary,path)
def write_csv(path,rows):
    rows=list(rows); columns=list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',newline='',dir=path.parent,delete=False) as handle: temporary=Path(handle.name); writer=csv.DictWriter(handle,fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary,path)
input_hashes_before={name:sha256_file(path) for name,path in INPUTS.items()}
write_json(OUTPUT_ROOT/'mmr_contract.json',CONTRACT)
print({'contract_frozen_before_results':True,'inputs':len(INPUTS),'lambdas':LAMBDAS,'depths':DEPTHS})


{'contract_frozen_before_results': True, 'inputs': 20, 'lambdas': (0.7, 0.8, 0.9), 'depths': (20, 50)}


In [2]:
chunks=read_jsonl(INPUTS['chunks_13']); chunk_by_id={row['id']:row for row in chunks}; assert len(chunks)==len(chunk_by_id)==327
query_source=read_csv(INPUTS['queries_13']); queries={}
for row in query_source:
    if row['query_id'] in queries:
        assert all(queries[row['query_id']][key]==row[key] for key in ('category','query','expected_card','expected_level','required_terms'))
    else: queries[row['query_id']]={key:row[key] for key in ('query_id','category','query','expected_card','expected_level','required_terms')}
assert len(queries)==30 and {category:sum(row['category']==category for row in queries.values()) for category in ('proper_noun','numeric_condition','semantic')}=={'proper_noun':10,'numeric_condition':10,'semantic':10}
relevance_rows=read_jsonl(INPUTS['relevance_18']); relevance_by_query={row['query_id']:row for row in relevance_rows}; assert set(relevance_by_query)==set(queries)
strict_by_query={qid:set(row['strict_chunk_ids']) for qid,row in relevance_by_query.items()}; answer_by_query={qid:set(row['answer_bearing_chunk_ids']) for qid,row in relevance_by_query.items()}
strict_groups={qid:set(row['strict_exact_group_ids']) for qid,row in relevance_by_query.items()}; answer_groups={qid:set(row['answer_bearing_exact_group_ids']) for qid,row in relevance_by_query.items()}
assert all(strict_by_query[qid] and strict_by_query[qid] <= answer_by_query[qid] for qid in queries)
group_rows=read_jsonl(INPUTS['exact_groups_18']); group_by_chunk={}
for row in group_rows:
    for chunk_id in row['member_chunk_ids']: assert chunk_id not in group_by_chunk; group_by_chunk[chunk_id]=row['group_id']
assert set(group_by_chunk)==set(chunk_by_id)
candidate_source=[row for row in read_csv(INPUTS['candidates_16']) if row['configuration']=='vector_0.4_bm25_0.6']; assert len(candidate_source)==30*50
candidates={}
for query_id in queries:
    rows=sorted((row for row in candidate_source if row['query_id']==query_id),key=lambda row:int(row['fused_rank']))
    assert len(rows)==50 and [int(row['fused_rank']) for row in rows]==list(range(1,51)) and len({row['chunk_id'] for row in rows})==50
    values=np.asarray([float(row['total_score']) for row in rows],dtype=np.float64); span=float(values.max()-values.min()); assert math.isfinite(span) and span>SPAN_EPS
    for row,value in zip(rows,(values-values.min())/span): row['rel_norm']=float(value)
    assert all(math.isfinite(row['rel_norm']) and 0<=row['rel_norm']<=1 for row in rows)
    candidates[query_id]=rows
assert all([row['chunk_id'] for row in candidates[qid][:20]]==[row['chunk_id'] for row in candidates[qid]][:20] for qid in queries)

embedding_items=[(f"chunk:{row['id']}",row['document']) for row in chunks]+[(f"query:{qid}",queries[qid]['query']) for qid in queries]
all_vectors=[]; offset=0
for name in CACHE_NAMES:
    cached=np.load(CACHE_ROOT/name,allow_pickle=False); size=len(cached['embeddings']); batch=embedding_items[offset:offset+size]; hashes=[sha256_text(text) for _,text in batch]
    assert cached.files==['embeddings','hashes'] and cached['embeddings'].dtype==np.float32 and cached['embeddings'].shape==(size,1536) and cached['hashes'].tolist()==hashes
    assert np.isfinite(cached['embeddings']).all(); all_vectors.append(cached['embeddings']); offset+=size
vectors=np.concatenate(all_vectors,axis=0); assert offset==len(embedding_items)==357 and vectors.shape==(357,1536)
chunk_vectors=vectors[:327].astype(np.float64); norms=np.linalg.norm(chunk_vectors,axis=1); assert np.isfinite(norms).all() and np.all(norms>0)
chunk_vectors/=norms[:,None]; assert np.allclose(np.linalg.norm(chunk_vectors,axis=1),1.0,atol=1e-12,rtol=0)
vector_by_id={chunk['id']:chunk_vectors[index] for index,chunk in enumerate(chunks)}; assert set(vector_by_id)==set(chunk_by_id)
for query_id,rows in candidates.items():
    matrix=np.clip(np.vstack([vector_by_id[row['chunk_id']] for row in rows])@np.vstack([vector_by_id[row['chunk_id']] for row in rows]).T,-1.0,1.0)
    assert np.isfinite(matrix).all() and np.allclose(matrix,matrix.T,atol=1e-12,rtol=0) and np.allclose(np.diag(matrix),1.0,atol=1e-12,rtol=0) and matrix.min()>=-1 and matrix.max()<=1
print({'chunks':len(chunks),'queries':len(queries),'candidates':len(candidate_source),'embeddings':vectors.shape,'float64_unit_norm':True,'cosine_contract':True})


{'chunks': 327, 'queries': 30, 'candidates': 1500, 'embeddings': (357, 1536), 'float64_unit_norm': True, 'cosine_contract': True}


In [3]:
def mmr_permutation(query_id,depth,lambda_value,emit_trace=False):
    rows=candidates[query_id][:depth]; ids=[row['chunk_id'] for row in rows]; rel={row['chunk_id']:row['rel_norm'] for row in rows}; rank={row['chunk_id']:int(row['fused_rank']) for row in rows}
    matrix=np.clip(np.vstack([vector_by_id[chunk_id] for chunk_id in ids])@np.vstack([vector_by_id[chunk_id] for chunk_id in ids]).T,-1.0,1.0); index={chunk_id:i for i,chunk_id in enumerate(ids)}
    selected=[ids[0]]; trace=[]
    if emit_trace: trace.append({'query_id':query_id,'depth':depth,'lambda':lambda_value,'selection_step':1,'chunk_id':ids[0],'original_rank':rank[ids[0]],'total_score':float(rows[0]['total_score']),'rel_norm':rel[ids[0]],'max_positive_similarity':0.0,'mmr_score':rel[ids[0]],'tie_candidate_count':1,'card_key':chunk_by_id[ids[0]]['metadata']['card_key'],'level':chunk_by_id[ids[0]]['metadata']['level']})
    while len(selected)<depth:
        scored=[]
        for chunk_id in ids:
            if chunk_id in selected: continue
            redundancy=max(0.0,max(float(matrix[index[chunk_id],index[chosen]]) for chosen in selected)); score=lambda_value*rel[chunk_id]-(1-lambda_value)*redundancy
            assert math.isfinite(redundancy) and math.isfinite(score); scored.append((chunk_id,score,redundancy))
        best=max(value[1] for value in scored); tied=[value for value in scored if best-value[1]<=TOLERANCE]; chosen,score,redundancy=min(tied,key=lambda value:(-rel[value[0]],rank[value[0]],value[0])); selected.append(chosen)
        if emit_trace:
            row=next(row for row in rows if row['chunk_id']==chosen); trace.append({'query_id':query_id,'depth':depth,'lambda':lambda_value,'selection_step':len(selected),'chunk_id':chosen,'original_rank':rank[chosen],'total_score':float(row['total_score']),'rel_norm':rel[chosen],'max_positive_similarity':redundancy,'mmr_score':score,'tie_candidate_count':len(tied),'card_key':chunk_by_id[chosen]['metadata']['card_key'],'level':chunk_by_id[chosen]['metadata']['level']})
    assert len(selected)==len(set(selected))==depth and set(selected)==set(ids)
    return selected,trace

rankings={}; step_trace=[]
for depth in DEPTHS:
    for query_id in queries:
        baseline=[row['chunk_id'] for row in candidates[query_id][:depth]]; rankings[(depth,'no_mmr',query_id)]=baseline
        lambda_one,_=mmr_permutation(query_id,depth,1.0); assert lambda_one==baseline
        for lambda_value in LAMBDAS:
            ranking,trace=mmr_permutation(query_id,depth,lambda_value,True); repeat,_=mmr_permutation(query_id,depth,lambda_value,False); assert repeat==ranking
            rankings[(depth,f'mmr_{lambda_value:.1f}',query_id)]=ranking; step_trace.extend(trace)
assert len(step_trace)==30*sum(DEPTHS)*len(LAMBDAS) and all(rankings[(20,system,qid)]==rankings[(50,system,qid)][:20] for system in ('no_mmr',) for qid in queries)
print({'rankings':len(rankings),'trace_rows':len(step_trace),'lambda_1_exact':True,'deterministic_repeat':True})


{'rankings': 240, 'trace_rows': 6300, 'lambda_1_exact': True, 'deterministic_repeat': True}


In [4]:
def compressed_group_ranking(raw_ranking): return list(dict.fromkeys(group_by_chunk[chunk_id] for chunk_id in raw_ranking))
def view_metrics(query_id,raw_ranking,view):
    query=queries[query_id]; card_hit=int(any(chunk_by_id[chunk_id]['metadata']['card_key']==query['expected_card'] for chunk_id in raw_ranking[:3]))
    if view=='strict_raw': units,relevant=raw_ranking,strict_by_query[query_id]
    elif view=='answer_bearing_raw': units,relevant=raw_ranking,answer_by_query[query_id]
    elif view=='strict_exact_doc_dedup': units,relevant=compressed_group_ranking(raw_ranking),strict_groups[query_id]
    elif view=='answer_bearing_exact_doc_dedup': units,relevant=compressed_group_ranking(raw_ranking),answer_groups[query_id]
    else: units,relevant=raw_ranking,{f'gold_family:{query_id}'}
    if view=='answer_bearing_gold_family_oracle':
        seen=False; hits=[]
        for chunk_id in units[:5]: gain=chunk_id in answer_by_query[query_id] and not seen; hits.append(gain); seen=seen or gain
    else: hits=[unit in relevant for unit in units[:5]]
    first=next((rank for rank,hit in enumerate(hits,1) if hit),None); dcg=sum(hit/math.log2(rank+1) for rank,hit in enumerate(hits,1)); ideal=sum(1/math.log2(rank+1) for rank in range(1,min(5,len(relevant))+1))
    result={'card_hit_at_3':card_hit,'hit_at_3':int(any(hits[:3])),'recall_at_5':sum(hits)/len(relevant),'mrr_at_5':1/first if first else 0.0,'ndcg_at_5':dcg/ideal,'relevant_unit_count':len(relevant),'ranking_unit_count':len(units)}
    assert all(0<=result[metric]<=1 for metric in METRICS)
    if view=='answer_bearing_gold_family_oracle': assert result['relevant_unit_count']==1 and result['recall_at_5'] in (0.0,1.0)
    return result
def in_group(row,group): return group=='all' or (group=='card' and row['question_group']=='card') or (group=='evidence' and row['question_group']=='evidence') or (group=='numeric' and row['category']=='numeric_condition') or (group=='semantic' and row['category']=='semantic')

per_query=[]
for depth in DEPTHS:
    for system in SYSTEMS:
        for query_id,query in queries.items():
            ranking=rankings[(depth,system,query_id)]; by_view={view:view_metrics(query_id,ranking,view) for view in VIEWS}; assert len({by_view[view]['card_hit_at_3'] for view in VIEWS})==1
            for view,values in by_view.items(): per_query.append({'configuration':'vector_0.4_bm25_0.6','depth':depth,'system':system,'lambda':'' if system=='no_mmr' else float(system.split('_')[1]),'query_id':query_id,'question_group':'card' if query['expected_level']=='card' else 'evidence','category':query['category'],'view':view,**values,'top5_chunk_ids':json.dumps(ranking[:5],separators=(',',':')),'top5_cards':json.dumps([chunk_by_id[i]['metadata']['card_key'] for i in ranking[:5]],ensure_ascii=False,separators=(',',':')),'top5_levels':json.dumps([chunk_by_id[i]['metadata']['level'] for i in ranking[:5]],separators=(',',':'))})
assert len(per_query)==2*4*30*5

baseline_18=[row for row in read_csv(INPUTS['per_query_18']) if row['comparison'] in {f'vector_0.4_bm25_0.6_top{depth}' for depth in DEPTHS} and row['system']=='no_reranker']; assert len(baseline_18)==2*30*5
baseline_lookup={(int(row['depth']),row['query_id'],row['view']):row for row in baseline_18}
for row in (row for row in per_query if row['system']=='no_mmr'):
    source=baseline_lookup[(row['depth'],row['query_id'],row['view'])]; assert row['top5_chunk_ids']==source['top5_chunk_ids'] and row['top5_cards']==source['top5_cards'] and row['top5_levels']==source['top5_levels'] and all(abs(float(row[m])-float(source[m]))<=TOLERANCE for m in METRICS)
summary=[]
for depth in DEPTHS:
    for system in SYSTEMS:
        for view in VIEWS:
            rows=[row for row in per_query if row['depth']==depth and row['system']==system and row['view']==view]
            for group in GROUPS:
                selected=[row for row in rows if in_group(row,group)]; assert len(selected)==DENOMINATORS[group]
                summary.append({'configuration':'vector_0.4_bm25_0.6','depth':depth,'system':system,'lambda':'' if system=='no_mmr' else float(system.split('_')[1]),'view':view,'group':group,'denominator':len(selected),**{metric:sum(float(row[metric]) for row in selected)/len(selected) for metric in METRICS}})
assert len(summary)==2*4*5*5
summary_18=[row for row in read_csv(INPUTS['summary_csv_18']) if row['comparison'] in {f'vector_0.4_bm25_0.6_top{depth}' for depth in DEPTHS} and row['system']=='no_reranker']; assert len(summary_18)==2*5*5
source_summary={(int(row['depth']),row['view'],row['group']):row for row in summary_18}
for row in (row for row in summary if row['system']=='no_mmr'): assert all(abs(float(row[m])-float(source_summary[(row['depth'],row['view'],row['group'])][m]))<=TOLERANCE for m in METRICS)
candidate_ceiling=[]
for depth in DEPTHS:
    for query_id,query in queries.items():
        pool=set(rankings[(depth,'no_mmr',query_id)]); strict=strict_by_query[query_id]; answer=answer_by_query[query_id]
        candidate_ceiling.append({'configuration':'vector_0.4_bm25_0.6','depth':depth,'query_id':query_id,'question_group':'card' if query['expected_level']=='card' else 'evidence','category':query['category'],'candidate_count':depth,'strict_relevant_count':len(strict),'strict_candidate_hit':int(bool(pool&strict)),'strict_candidate_recall':len(pool&strict)/len(strict),'answer_bearing_relevant_count':len(answer),'answer_bearing_candidate_hit':int(bool(pool&answer)),'answer_bearing_candidate_recall':len(pool&answer)/len(answer),'gold_family_candidate_recall':float(bool(pool&answer)),'expected_card_candidate_count':sum(chunk_by_id[i]['metadata']['card_key']==query['expected_card'] for i in pool),'missing_strict_ids':json.dumps(sorted(strict-pool),separators=(',',':')),'missing_answer_bearing_ids':json.dumps(sorted(answer-pool),separators=(',',':'))})
assert len(candidate_ceiling)==60
print({'per_query':len(per_query),'summary':len(summary),'candidate_ceiling':len(candidate_ceiling),'baseline_18_exact':True})


{'per_query': 1200, 'summary': 200, 'candidate_ceiling': 60, 'baseline_18_exact': True}


In [5]:
def redundancy_diagnostics(depth,system,query_id):
    top=rankings[(depth,system,query_id)][:5]; query=queries[query_id]; candidate_rows=candidates[query_id][:depth]; original={row['chunk_id']:int(row['fused_rank']) for row in candidate_rows}; relevance={row['chunk_id']:row['rel_norm'] for row in candidate_rows}
    pair_values=[]; cross_level=0; containment=0; cross_high=0; level_values={}
    nearest={chunk_id:[] for chunk_id in top}
    for left,right in itertools.combinations(top,2):
        cosine=float(np.clip(vector_by_id[left]@vector_by_id[right],-1.0,1.0)); positive=max(0.0,cosine); pair_values.append(positive); nearest[left].append(positive); nearest[right].append(positive)
        lm,rm=chunk_by_id[left]['metadata'],chunk_by_id[right]['metadata']; level_key='|'.join(sorted((lm['level'],rm['level']))); level_values.setdefault(level_key,[]).append(cosine)
        if lm['card_key']==rm['card_key'] and lm['level']!=rm['level']:
            cross_level+=1; left_text=normalize(chunk_by_id[left]['document']); right_text=normalize(chunk_by_id[right]['document']); contains=bool(left_text and right_text and (left_text in right_text or right_text in left_text)); containment+=int(contains); cross_high+=int(cosine>=0.90)
    groups=[group_by_chunk[i] for i in top]; cards=[chunk_by_id[i]['metadata']['card_key'] for i in top]; expected_count=sum(card==query['expected_card'] for card in cards)
    return {'configuration':'vector_0.4_bm25_0.6','depth':depth,'system':system,'lambda':'' if system=='no_mmr' else float(system.split('_')[1]),'query_id':query_id,'question_group':'card' if query['expected_level']=='card' else 'evidence','category':query['category'],'positive_pairwise_cosine_mean':statistics.fmean(pair_values),'positive_pairwise_cosine_max':max(pair_values),'nearest_neighbor_positive_cosine_mean':statistics.fmean(max(values) for values in nearest.values()),'exact_duplicate_excess':len(top)-len(set(groups)),'exact_unique_group_count':len(set(groups)),'same_card_cross_level_pair_count':cross_level,'cross_level_containment_pair_count':containment,'cross_level_cosine_ge_0_90_pair_count':cross_high,'level_pair_mean_cosine':json.dumps({key:statistics.fmean(values) for key,values in sorted(level_values.items())},sort_keys=True,separators=(',',':')),'unique_card_count':len(set(cards)),'expected_card_count':expected_count,'off_card_count':len(top)-expected_count,'expected_card_share_at_5':expected_count/5,'original_rank_mean':statistics.fmean(original[i] for i in top),'original_rank_max':max(original[i] for i in top),'rel_norm_mean':statistics.fmean(relevance[i] for i in top),'rel_norm_sum':sum(relevance[i] for i in top),'top5_chunk_ids':json.dumps(top,separators=(',',':'))}

redundancy_per_query=[redundancy_diagnostics(depth,system,query_id) for depth in DEPTHS for system in SYSTEMS for query_id in queries]; assert len(redundancy_per_query)==240
baseline_diag={(row['depth'],row['query_id']):row for row in redundancy_per_query if row['system']=='no_mmr'}
for row in redundancy_per_query:
    base=baseline_diag[(row['depth'],row['query_id'])]
    row['original_rank_mean_delta']=row['original_rank_mean']-base['original_rank_mean']; row['rel_norm_mean_delta']=row['rel_norm_mean']-base['rel_norm_mean']
    if row['system']=='no_mmr': row['redundancy_outcome']='baseline'
    else:
        diagnostics=('exact_duplicate_excess','cross_level_containment_pair_count','positive_pairwise_cosine_mean'); deltas=[row[name]-base[name] for name in diagnostics]
        row['redundancy_outcome']='worsened' if any(delta>TOLERANCE for delta in deltas) else ('improved' if any(delta < -TOLERANCE for delta in deltas) else 'tie')

REDUNDANCY_METRICS=('positive_pairwise_cosine_mean','positive_pairwise_cosine_max','nearest_neighbor_positive_cosine_mean','exact_duplicate_excess','exact_unique_group_count','same_card_cross_level_pair_count','cross_level_containment_pair_count','cross_level_cosine_ge_0_90_pair_count','unique_card_count','expected_card_count','off_card_count','expected_card_share_at_5','original_rank_mean','original_rank_max','rel_norm_mean','rel_norm_sum','original_rank_mean_delta','rel_norm_mean_delta')
redundancy_summary=[]
for depth in DEPTHS:
    for system in SYSTEMS:
        for group in GROUPS:
            selected=[row for row in redundancy_per_query if row['depth']==depth and row['system']==system and in_group(row,group)]; assert len(selected)==DENOMINATORS[group]
            counts={name:sum(row['redundancy_outcome']==name for row in selected) for name in ('improved','worsened','tie')}
            redundancy_summary.append({'configuration':'vector_0.4_bm25_0.6','depth':depth,'system':system,'lambda':'' if system=='no_mmr' else float(system.split('_')[1]),'group':group,'denominator':len(selected),**{name:statistics.fmean(float(row[name]) for row in selected) for name in REDUNDANCY_METRICS},'redundancy_improved':counts['improved'],'redundancy_worsened':counts['worsened'],'redundancy_tied':counts['tie']})
assert len(redundancy_summary)==40
level_pair_rows=[]
for row in redundancy_per_query:
    for level_pair,value in json.loads(row['level_pair_mean_cosine']).items(): level_pair_rows.append({'depth':row['depth'],'system':row['system'],'lambda':row['lambda'],'query_id':row['query_id'],'question_group':row['question_group'],'category':row['category'],'level_pair':level_pair,'mean_cosine':value})
assert all(math.isfinite(float(row['mean_cosine'])) and -1<=float(row['mean_cosine'])<=1 for row in level_pair_rows)
print({'redundancy_per_query':len(redundancy_per_query),'redundancy_summary':len(redundancy_summary),'level_pair_rows':len(level_pair_rows)})


{'redundancy_per_query': 240, 'redundancy_summary': 40, 'level_pair_rows': 1218}


In [6]:
per_query_lookup={(row['depth'],row['system'],row['query_id'],row['view']):row for row in per_query}
paired=[]
for depth in DEPTHS:
    for lambda_value in LAMBDAS:
        system=f'mmr_{lambda_value:.1f}'
        for query_id in queries:
            for view in VIEWS:
                current=per_query_lookup[(depth,system,query_id,view)]; base=per_query_lookup[(depth,'no_mmr',query_id,view)]; row={'configuration':'vector_0.4_bm25_0.6','depth':depth,'system':system,'lambda':lambda_value,'query_id':query_id,'question_group':current['question_group'],'category':current['category'],'view':view}
                for metric in METRICS:
                    delta=float(current[metric])-float(base[metric]); row[f'mmr_{metric}']=current[metric]; row[f'baseline_{metric}']=base[metric]; row[f'{metric}_delta']=delta; row[f'{metric}_outcome']='win' if delta>TOLERANCE else ('loss' if delta < -TOLERANCE else 'tie')
                paired.append(row)
assert len(paired)==2*3*30*5
wlt=[]
for depth in DEPTHS:
    for lambda_value in LAMBDAS:
        system=f'mmr_{lambda_value:.1f}'
        for view in VIEWS:
            for group in GROUPS:
                rows=[row for row in paired if row['depth']==depth and row['system']==system and row['view']==view and in_group(row,group)]; assert len(rows)==DENOMINATORS[group]
                for metric in METRICS:
                    counts={outcome:sum(row[f'{metric}_outcome']==outcome for row in rows) for outcome in ('win','loss','tie')}; assert sum(counts.values())==DENOMINATORS[group]
                    wlt.append({'depth':depth,'system':system,'lambda':lambda_value,'view':view,'group':group,'metric':metric,'denominator':len(rows),'wins':counts['win'],'losses':counts['loss'],'ties':counts['tie'],'mean_delta':statistics.fmean(float(row[f'{metric}_delta']) for row in rows)})
assert len(wlt)==2*3*5*5*5

summary_lookup={(row['depth'],row['system'],row['view'],row['group']):row for row in summary}; redundancy_lookup={(row['depth'],row['system'],row['group']):row for row in redundancy_summary}
def ge(current,baseline): return current+TOLERANCE>=baseline
gate_results={}
for lambda_value in LAMBDAS:
    system=f'mmr_{lambda_value:.1f}'; depth=PRIMARY_DEPTH; checks={}
    strict=summary_lookup[(depth,system,'strict_raw','evidence')]; strict_base=summary_lookup[(depth,'no_mmr','strict_raw','evidence')]
    for metric in METRICS: checks[f'strict_raw_evidence_{metric}_nonregression']=ge(strict[metric],strict_base[metric])
    checks['strict_raw_evidence_mrr_or_ndcg_strict_improvement']=strict['mrr_at_5']>strict_base['mrr_at_5']+TOLERANCE or strict['ndcg_at_5']>strict_base['ndcg_at_5']+TOLERANCE
    for view,metrics in (('answer_bearing_raw',('hit_at_3','mrr_at_5')),('strict_exact_doc_dedup',('hit_at_3','mrr_at_5','ndcg_at_5')),('answer_bearing_exact_doc_dedup',('hit_at_3','mrr_at_5','ndcg_at_5')),('answer_bearing_gold_family_oracle',('hit_at_3','mrr_at_5','ndcg_at_5'))):
        current=summary_lookup[(depth,system,view,'evidence')]; base=summary_lookup[(depth,'no_mmr',view,'evidence')]
        for metric in metrics: checks[f'{view}_evidence_{metric}_nonregression']=ge(current[metric],base[metric])
    checks['card10_card_hit_at_3_nonregression']=ge(summary_lookup[(depth,system,'strict_raw','card')]['card_hit_at_3'],summary_lookup[(depth,'no_mmr','strict_raw','card')]['card_hit_at_3'])
    diversity=redundancy_lookup[(depth,system,'evidence')]; diversity_base=redundancy_lookup[(depth,'no_mmr','evidence')]
    checks['evidence_expected_card_share_nonregression']=ge(diversity['expected_card_share_at_5'],diversity_base['expected_card_share_at_5'])
    checks['exact_duplicate_excess_nonincrease']=diversity['exact_duplicate_excess']<=diversity_base['exact_duplicate_excess']+TOLERANCE
    checks['cross_level_containment_nonincrease']=diversity['cross_level_containment_pair_count']<=diversity_base['cross_level_containment_pair_count']+TOLERANCE
    checks['cosine_or_containment_strict_decrease']=diversity['positive_pairwise_cosine_mean']<diversity_base['positive_pairwise_cosine_mean']-TOLERANCE or diversity['cross_level_containment_pair_count']<diversity_base['cross_level_containment_pair_count']-TOLERANCE
    checks['redundancy_wlt_improved_gt_worsened']=diversity['redundancy_improved']>diversity['redundancy_worsened']
    gate_results[system]={'lambda':lambda_value,'eligible':all(checks.values()),'checks':checks,'strict_raw_evidence':{metric:strict[metric] for metric in METRICS},'strict_raw_evidence_deltas':{metric:strict[metric]-strict_base[metric] for metric in METRICS},'evidence_diversity':{name:diversity[name] for name in ('expected_card_share_at_5','exact_duplicate_excess','cross_level_containment_pair_count','positive_pairwise_cosine_mean','redundancy_improved','redundancy_worsened','redundancy_tied')},'evidence_diversity_deltas':{name:diversity[name]-diversity_base[name] for name in ('expected_card_share_at_5','exact_duplicate_excess','cross_level_containment_pair_count','positive_pairwise_cosine_mean')}}
eligible=[system for system,result in gate_results.items() if result['eligible']]
if eligible:
    selected_system=min(eligible,key=lambda system:(-summary_lookup[(50,system,'strict_raw','evidence')]['ndcg_at_5'],-summary_lookup[(50,system,'strict_raw','evidence')]['mrr_at_5'],-redundancy_lookup[(50,system,'evidence')]['expected_card_share_at_5'],redundancy_lookup[(50,system,'evidence')]['cross_level_containment_pair_count'],redundancy_lookup[(50,system,'evidence')]['positive_pairwise_cosine_mean'],-float(system.split('_')[1]))); decision='select_'+selected_system
else: selected_system='no_mmr'; decision='retain_no_mmr'
selection_decision={'decision':decision,'selected_system':selected_system,'selected_lambda':None if selected_system=='no_mmr' else float(selected_system.split('_')[1]),'primary_depth':50,'eligible_lambdas':[gate_results[system]['lambda'] for system in eligible],'gate_results':gate_results,'top20_interpretation':'efficiency supplement for the primary-selected lambda only; not separately selected','promotion_scope':'development candidate only'}
primary_rows=[row for row in summary if row['depth']==50 and row['group'] in ('evidence','numeric','semantic') and row['view'] in ('strict_raw','answer_bearing_raw')]
summary_json={'schema_version':'mmr_redundancy_ablation_summary_v1','contract':CONTRACT,'selection':selection_decision,'primary_rows':primary_rows,'wlt':wlt,'row_counts':{'trace':len(step_trace),'per_query_metrics':len(per_query),'summary':len(summary),'paired_deltas':len(paired),'wlt':len(wlt),'redundancy_per_query':len(redundancy_per_query),'redundancy_summary':len(redundancy_summary),'candidate_ceiling':len(candidate_ceiling),'level_pair_cosine':len(level_pair_rows)},'limitations':['single development-set run','MMR cannot recover chunks outside stored Top20/Top50','embedding cosine is a redundancy proxy, not factual equivalence','containment is exact normalized substring only','Top20 is interpreted only for the Top50-selected lambda']}
print({'paired':len(paired),'wlt':len(wlt),'decision':decision,'eligible':eligible})


{'paired': 900, 'wlt': 750, 'decision': 'retain_no_mmr', 'eligible': []}


In [7]:
write_csv(OUTPUT_ROOT/'mmr_step_trace.csv',step_trace); write_csv(OUTPUT_ROOT/'mmr_per_query_metrics.csv',per_query); write_csv(OUTPUT_ROOT/'mmr_summary.csv',summary); write_json(OUTPUT_ROOT/'mmr_summary.json',summary_json)
write_csv(OUTPUT_ROOT/'mmr_paired_deltas.csv',paired); write_csv(OUTPUT_ROOT/'mmr_wlt.csv',wlt); write_csv(OUTPUT_ROOT/'mmr_redundancy_per_query.csv',redundancy_per_query); write_csv(OUTPUT_ROOT/'mmr_redundancy_summary.csv',redundancy_summary); write_csv(OUTPUT_ROOT/'mmr_level_pair_cosine.csv',level_pair_rows); write_csv(OUTPUT_ROOT/'mmr_candidate_ceiling.csv',candidate_ceiling); write_json(OUTPUT_ROOT/'mmr_selection_decision.json',selection_decision)
readme=f'''# 19 MMR redundancy ablation

Codex coder agent가 `skn25` fresh kernel, CPU/offline으로 실행한 개발셋 단일 진단이다. 저장된 0.4:0.6 RRF 후보와 chunk embedding만 재사용했다.

- MMR(Maximal Marginal Relevance): 관련성은 유지하면서 이미 선택한 청크와 비슷한 청크에 벌점을 주는 순위 선택법
- redundancy(중복성): Top5 청크 사이 양의 cosine, exact duplicate, 같은 카드의 계층 간 포함 관계로 진단
- exact-document dedup: 카드와 정규화 문서가 완전히 같은 청크를 한 묶음으로 보는 평가
- answer-bearing(level-relaxed term-bearing): 기대 카드와 필수 용어를 포함하지만 level을 무시한 진단이며 사실적으로 완전한 정답과 같지 않다
- W/L/T: 같은 질의의 no-MMR 대비 승/패/동률 수
- decision: `{decision}`; primary Top50 gate 통과 λ: {selection_decision['eligible_lambdas']}
- Top20은 primary에서 선택된 같은 λ의 효율성 보조 비교로만 해석한다.

후보 밖 recall을 개선할 수 없고 cosine은 의미 중복의 대리 지표다. 정확 포함만 세므로 표현이 다른 중복은 놓친다. 개발셋 결과이며 holdout·운영 일반화를 주장하지 않는다. GPU/model/custom code/network/API/new embedding/Chroma/package install 호출은 모두 0이다.
'''
(OUTPUT_ROOT/'README.md').write_text(readme,encoding='utf-8')
input_hashes_after={name:sha256_file(path) for name,path in INPUTS.items()}; assert input_hashes_after==input_hashes_before
output_names=('mmr_contract.json','mmr_step_trace.csv','mmr_per_query_metrics.csv','mmr_summary.csv','mmr_summary.json','mmr_paired_deltas.csv','mmr_wlt.csv','mmr_redundancy_per_query.csv','mmr_redundancy_summary.csv','mmr_level_pair_cosine.csv','mmr_candidate_ceiling.csv','mmr_selection_decision.json','README.md')
integrity={'schema_version':'mmr_redundancy_integrity_v1','self_hash_excluded':True,'inputs':{name:{'path':str(path.relative_to(PROJECT_ROOT)),'sha256_before':input_hashes_before[name],'sha256_after':input_hashes_after[name],'unchanged':True} for name,path in INPUTS.items()},'outputs':{name:{'sha256':sha256_file(OUTPUT_ROOT/name),'bytes':(OUTPUT_ROOT/name).stat().st_size} for name in output_names},'notebook':{'path':'notebooks/19_mmr_redundancy_ablation.ipynb','sha256':'pending_after_nbclient_serialization'},'row_counts':summary_json['row_counts'],'assertions':{'chunk_query_counts':True,'top20_prefix':True,'top50_anchor_float64':True,'embedding_cache_hash_text_coverage':True,'embedding_float32_finite':True,'unit_norm_float64':True,'cosine_symmetric_diagonal_range':True,'lambda_1_fused_rank_exact':True,'permutation_unique_deterministic':True,'metrics_and_card_invariants':True,'baseline_18_per_query_top5_summary_exact':True,'candidate_ceiling_depth_common':True,'wlt_denominators':True,'source_hashes_unchanged':True},'execution':CONTRACT['execution']}
write_json(OUTPUT_ROOT/'mmr_integrity.json',integrity)
expected_counts={'mmr_step_trace.csv':6300,'mmr_per_query_metrics.csv':1200,'mmr_summary.csv':200,'mmr_paired_deltas.csv':900,'mmr_wlt.csv':750,'mmr_redundancy_per_query.csv':240,'mmr_redundancy_summary.csv':40,'mmr_candidate_ceiling.csv':60}
for name,count in expected_counts.items(): assert len(read_csv(OUTPUT_ROOT/name))==count
assert all(int(row['wins'])+int(row['losses'])+int(row['ties'])==int(row['denominator']) for row in read_csv(OUTPUT_ROOT/'mmr_wlt.csv'))
assert all(math.isfinite(float(row[metric])) and 0<=float(row[metric])<=1 for row in read_csv(OUTPUT_ROOT/'mmr_per_query_metrics.csv') for metric in METRICS)
assert {name:sha256_file(path) for name,path in INPUTS.items()}==input_hashes_before
stored_integrity=json.loads((OUTPUT_ROOT/'mmr_integrity.json').read_text(encoding='utf-8')); assert stored_integrity['self_hash_excluded'] and all(item['unchanged'] for item in stored_integrity['inputs'].values())
print({'validation':'PASS','rows':expected_counts,'decision':decision,'inputs_unchanged':True,'gpu_model_network_chroma_calls':0})


{'validation': 'PASS', 'rows': {'mmr_step_trace.csv': 6300, 'mmr_per_query_metrics.csv': 1200, 'mmr_summary.csv': 200, 'mmr_paired_deltas.csv': 900, 'mmr_wlt.csv': 750, 'mmr_redundancy_per_query.csv': 240, 'mmr_redundancy_summary.csv': 40, 'mmr_candidate_ceiling.csv': 60}, 'decision': 'retain_no_mmr', 'inputs_unchanged': True, 'gpu_model_network_chroma_calls': 0}
